# Analise Exploratoria de Dados - Fingerprint de Trafego Wi-Fi IoT

Este notebook investiga o dataset tratado (`data/02 - Tratados/processed_training2.csv`) **antes** das
decisoes de engenharia de features e modelagem ja consolidadas no projeto. O objetivo e responder,
com evidencia, as perguntas que justificam as escolhas feitas no pipeline (`src/iot_fingerprint/`):

1. O dataset tem qualidade suficiente (poucos nulos, tipos consistentes)?
2. Quao desbalanceado e o conjunto entre dispositivos?
3. As variaveis brutas (tamanho de frame, IAT, power management, destinos) realmente diferenciam dispositivos?
4. O agregado em janelas de 100 pacotes e uma escolha razoavel, ou outro tamanho seria melhor?
5. Apos a agregacao, os dispositivos ficam separaveis no espaco de features?

As conclusoes de cada secao alimentam diretamente as decisoes documentadas em `reports/academic_report.md`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from iot_fingerprint.config import PROCESSED_TRAINING_CSV
from iot_fingerprint.data import load_processed_training
from iot_fingerprint.features import build_device_windows

sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 30)
plt.rcParams["figure.dpi"] = 110

## 1. Visao geral e qualidade dos dados

Antes de qualquer tratamento, olhamos para o CSV bruto (ja extraido das capturas, mas sem o `dropna`
aplicado em `load_processed_training`) para medir o impacto da limpeza.

In [ ]:
raw = pd.read_csv(PROCESSED_TRAINING_CSV)
print(f"Linhas brutas: {len(raw):,}")
print(f"Colunas: {list(raw.columns)}")
print()
print("Valores nulos por coluna:")
print(raw.isna().sum())
print()
print("Tipos originais:")
print(raw.dtypes)

In [ ]:
df = load_processed_training()
dropped = len(raw) - len(df)
print(f"Linhas apos tratamento (load_processed_training): {len(df):,}")
print(f"Linhas removidas por valores ausentes: {dropped} ({dropped / len(raw):.3%} do total)")
df.head()

**Leitura:** a perda de dados pelo `dropna` e marginal (bem menos de 1% das linhas), concentrada na
primeira linha de cada captura (sem `time_delta_displayed`) e em poucos frames sem MAC de origem/destino
resolvido. Isso confirma que o tratamento atual em `data.py` e seguro e nao introduz vies relevante.

## 2. Desbalanceamento entre dispositivos

O alvo do modelo e o MAC de origem (`wlan.sa`). Antes de modelar, precisamos entender o quao desigual
e a distribuicao de frames entre os 15 dispositivos observados.

In [ ]:
device_counts = df["wlan.sa"].value_counts()
display(device_counts.describe())

plt.figure(figsize=(10, 6))
device_counts.sort_values().plot(kind="barh", color="#2a9d8f")
plt.title("Frames por dispositivo (MAC de origem)")
plt.xlabel("Frames")
plt.ylabel("MAC de origem")
plt.tight_layout()
plt.show()

In [ ]:
imbalance_ratio = device_counts.max() / device_counts.min()
print(f"Dispositivo mais ativo: {device_counts.idxmax()} ({device_counts.max():,} frames)")
print(f"Dispositivo menos ativo: {device_counts.idxmin()} ({device_counts.min():,} frames)")
print(f"Razao entre o mais e o menos ativo: {imbalance_ratio:.1f}x")

**Leitura:** ha uma razao de desbalanceamento de mais de 1000x entre o dispositivo mais e o menos
ativo. Isso justifica diretamente duas decisoes do pipeline de modelagem: o uso de **Macro F1** (em vez
de accuracy pura) como criterio de tuning/selecao, e o `class_weight="balanced"` usado no Random Forest e
no SVM (`src/model_specs.py`). Tambem reforca por que dispositivos com poucas janelas agregadas
(`MIN_WINDOWS_PER_DEVICE = 10`) precisam ser descartados do treino: com tao pouca amostra, qualquer
validacao para essas classes seria estatisticamente fragil.

## 3. Tamanho de frame por dispositivo

`frame.len` e uma das features centrais. Verificamos sua distribuicao geral e por dispositivo.

In [ ]:
print(df["frame.len"].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df["frame.len"], bins=60, color="#457b9d")
plt.title("Distribuicao geral do tamanho de frame")
plt.xlabel("frame.len")
plt.tight_layout()
plt.show()

In [ ]:
top_devices = device_counts.head(8).index
sample = (
    df[df["wlan.sa"].isin(top_devices)]
    .groupby("wlan.sa", group_keys=False)
    .apply(lambda rows: rows.sample(min(len(rows), 3000), random_state=42))
)

plt.figure(figsize=(12, 6))
sns.boxplot(data=sample, x="wlan.sa", y="frame.len", order=top_devices)
plt.title("Tamanho de frame nos 8 dispositivos mais ativos")
plt.xlabel("MAC de origem")
plt.ylabel("frame.len")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

**Leitura:** a distribuicao geral e fortemente assimetrica, com mediana baixa (muitos frames de
controle/gerenciamento pequenos) e uma cauda longa de frames de dados maiores. Por dispositivo, ha
diferencas visiveis tanto na mediana quanto na dispersao do `frame.len`, o que sustenta empiricamente o
uso de `frame_len_mean`, `frame_len_std`, `frame_len_min` e `frame_len_max` como features (`features.py`) —
um unico ponto de tamanho de frame nao seria suficiente, mas a combinacao de momentos estatisticos
captura o perfil de cada dispositivo.

## 4. Intervalo entre pacotes (IAT)

`frame.time_delta_displayed` mede o tempo entre frames consecutivos. Cadencias de transmissao tendem a
ser uma assinatura forte de dispositivos IoT (ex.: beacons periodicos vs. trafego em rajada).

In [ ]:
print(df["frame.time_delta_displayed"].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df["frame.time_delta_displayed"].clip(upper=df["frame.time_delta_displayed"].quantile(0.99)),
             bins=60, color="#e76f51")
plt.title("Distribuicao do IAT (cauda de 1% removida para visualizacao)")
plt.xlabel("frame.time_delta_displayed (s)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
iat_clip = sample["frame.time_delta_displayed"].clip(upper=sample["frame.time_delta_displayed"].quantile(0.99))
sns.boxplot(x=sample["wlan.sa"], y=iat_clip, order=top_devices)
plt.title("IAT por dispositivo (top 8, cauda de 1% removida)")
plt.xlabel("MAC de origem")
plt.ylabel("frame.time_delta_displayed (s)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

**Leitura:** o IAT tambem varia visivelmente entre dispositivos, com alguns mostrando cadencias mais
regulares (caixa estreita) e outros com trafego mais irregular (caixa larga, mais outliers). Isso confirma
que `iat_mean`, `iat_std`, `iat_min` e `iat_max` carregam sinal discriminativo, alinhado ao que o relatorio
academico (secao 5) reporta como "intervalo medio entre pacotes" relevante.

## 5. Power management e destinos unicos

`wlan.fc.pwrmgt` indica se o dispositivo sinaliza economia de energia, e `wlan.da` (destino) reflete o
padrao de comunicacao (ex.: sempre falando com o mesmo AP vs. multiplos destinos).

In [ ]:
pwrmgt_by_device = df.groupby("wlan.sa")["wlan.fc.pwrmgt"].mean().sort_values()
plt.figure(figsize=(10, 6))
pwrmgt_by_device.plot(kind="barh", color="#6a4c93")
plt.title("Proporcao de frames com power management ativo, por dispositivo")
plt.xlabel("Proporcao pwrmgt=1")
plt.tight_layout()
plt.show()

In [ ]:
dest_by_device = df.groupby("wlan.sa")["wlan.da"].nunique().sort_values()
plt.figure(figsize=(10, 6))
dest_by_device.plot(kind="barh", color="#f4a261")
plt.title("Quantidade de destinos distintos por dispositivo (trafego completo)")
plt.xlabel("Destinos unicos")
plt.tight_layout()
plt.show()

**Leitura:** a proporcao de power management varia fortemente entre dispositivos (de quase 0 a perto
de 1), o que e esperado dado que dispositivos IoT de bateria tendem a sinalizar economia de energia com
mais frequencia que dispositivos sempre ligados. A quantidade de destinos distintos tambem varia, mas e
preciso cautela: como ha apenas 38 destinos unicos no dataset inteiro (provavelmente APs/gateways da
mesma captura), essa feature pode estar mais ligada a topologia especifica da captura do que a uma
caracteristica intrinseca do dispositivo — risco ja registrado na limitacao 4 do relatorio academico
(generalizacao para outros ambientes).

## 6. Correlacao entre variaveis brutas

Antes de agregar em janelas, verificamos se ha colinearidade forte entre as variaveis numericas brutas
que pudesse sugerir redundancia.

In [ ]:
raw_numeric = df[["frame.len", "frame.time_delta_displayed", "wlan.fc.pwrmgt"]].astype(float)
corr = raw_numeric.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="crest", square=True)
plt.title("Correlacao entre variaveis brutas")
plt.tight_layout()
plt.show()

**Leitura:** as variaveis brutas tem correlacao fraca entre si, o que indica que cada uma contribui
com informacao distinta e justifica manter as tres como base da engenharia de features, em vez de
descartar alguma por redundancia.

## 7. Escolha do tamanho de janela

O pipeline agrega o trafego em janelas fixas de 100 pacotes por dispositivo (`build_device_windows`,
`window_size=100`). Aqui comparamos esse valor com alternativas para entender o trade-off entre
estabilidade estatistica (janelas maiores, menos ruido) e quantidade de amostras disponiveis para
treino/validacao (janelas menores, mais janelas).

In [ ]:
window_sizes = [25, 50, 100, 200, 500]
rows = []
for size in window_sizes:
    windows = build_device_windows(df, window_size=size)
    counts = windows["wlan.sa"].value_counts()
    valid = counts[counts >= 10]
    rows.append(
        {
            "window_size": size,
            "total_windows": len(windows),
            "devices_with_10+_windows": len(valid),
            "min_windows_per_device": int(counts.min()),
            "median_windows_per_device": int(counts.median()),
            "frame_len_mean_std_across_windows": windows.groupby("wlan.sa")["frame_len_mean"].std().mean(),
        }
    )

window_summary = pd.DataFrame(rows)
window_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(window_summary["window_size"], window_summary["total_windows"], marker="o", color="#2a9d8f")
axes[0].set_title("Total de janelas vs. tamanho da janela")
axes[0].set_xlabel("Pacotes por janela")
axes[0].set_ylabel("Total de janelas")

axes[1].plot(window_summary["window_size"], window_summary["devices_with_10+_windows"], marker="o", color="#e76f51")
axes[1].set_title("Dispositivos com >=10 janelas vs. tamanho da janela")
axes[1].set_xlabel("Pacotes por janela")
axes[1].set_ylabel("Dispositivos validos")
plt.tight_layout()
plt.show()

**Leitura:** janelas menores (25-50 pacotes) geram muito mais amostras, mas comecam a excluir do
treino os dispositivos menos ativos por nao atingirem o minimo de 10 janelas (`MIN_WINDOWS_PER_DEVICE`).
Janelas muito grandes (500) reduzem demais o numero de janelas para os dispositivos com menos trafego,
comprometendo a validacao cruzada e o holdout temporal. O valor de 100 pacotes adotado no projeto fica
no meio dessa curva: preserva todos os 15 dispositivos com amostra minima razoavel e ainda gera milhares
de janelas no total. Essa analise da suporte empirico, e nao apenas argumentativo, a escolha de
`window_size=100` em `features.py`.

## 8. Separabilidade dos dispositivos apos a agregacao em janelas

Por fim, verificamos se a fingerprint agregada (janelas de 100 pacotes, a mesma usada no treino) produz
de fato grupos visualmente separaveis no espaco de features — o pre-requisito basico para qualquer
classificador funcionar bem.

In [ ]:
features = build_device_windows(df, window_size=100)
counts = features["wlan.sa"].value_counts()
features = features[features["wlan.sa"].isin(counts[counts >= 10].index)].copy()
print(f"Janelas usadas (mesmo filtro do pipeline de treino): {len(features)}")
features.describe()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

feature_cols = [
    "packet_count", "frame_len_mean", "frame_len_std", "frame_len_min", "frame_len_max",
    "iat_mean", "iat_std", "iat_min", "iat_max", "pwrmgt_ratio", "unique_destinations",
]
x_scaled = StandardScaler().fit_transform(features[feature_cols])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(x_scaled)

plot_df = features[["wlan.sa"]].copy()
plot_df["pc1"], plot_df["pc2"] = coords[:, 0], coords[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_df, x="pc1", y="pc2", hue="wlan.sa", alpha=0.6, palette="tab20")
plt.title("PCA (2 componentes) das janelas agregadas, por dispositivo")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Variancia explicada pelas 2 componentes: {pca.explained_variance_ratio_.sum():.1%}")

**Leitura:** mesmo em apenas 2 componentes principais, varios dispositivos formam aglomerados bem
distintos, o que confirma visualmente o que os modelos treinados (Random Forest, KNN, SVM) capturam
numericamente com Macro F1 acima de 0.91 em todos os protocolos de validacao. Os agrupamentos que
aparecem mais proximos ou misturados na projecao 2D sao bons candidatos a explicar os erros de
classificacao mais frequentes reportados nas matrizes de confusao (`reports/figures/*_confusion_matrix.png`).

## 9. Conclusoes da EDA

1. **Qualidade dos dados:** o tratamento atual (`data.py`) descarta menos de 1% das linhas; nao ha
   problema de qualidade que exija revisao.
2. **Desbalanceamento severo** (>1000x entre dispositivos) justifica o uso de Macro F1, `class_weight`
   balanceado e o filtro de janelas minimas por dispositivo ja presentes no pipeline.
3. **Tamanho de frame e IAT** mostram diferencas claras entre dispositivos tanto em nivel de frame
   individual quanto agregado, sustentando a escolha dessas variaveis como base da fingerprint.
4. **Power management** e fortemente diferenciador entre dispositivos; **destinos unicos** tambem ajuda,
   mas com a ressalva de que pode refletir a topologia da captura, nao so o dispositivo em si.
5. **Janela de 100 pacotes** e uma escolha apoiada por dados: equilibra estabilidade estatistica e volume
   de amostras disponiveis para todos os 15 dispositivos.
6. **Apos a agregacao**, os dispositivos formam aglomerados visualmente identificaveis mesmo em 2D (PCA),
   o que explica a alta performance dos classificadores treinados e reforca que a etapa de modelagem nao
   está superajustando a ruido — ha sinal real nos dados.

Essas conclusoes nao alteram o pipeline existente, mas documentam e fundamentam, com evidencia
exploratoria, as decisoes de design ja tomadas em `src/iot_fingerprint/features.py` e
`src/iot_fingerprint/model_pipeline.py`.